# Phase 1B: Linear SD with Better Draft/Target Ratio
**Purpose:** Contrast Phase 1 (1B/3B = 0.33 ratio, slowdown) with a proper ratio (1B/8B = 0.125).

Uses Llama-3.1-8B (4-bit quantized, ~5GB) as target + Llama-3.2-1B (fp16, ~2GB) as draft.
Same tokenizer family, no UAG needed, fits on T4.


In [1]:
!pip install -q transformers accelerate bitsandbytes matplotlib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 8.5 MB/s eta 0:00:00


In [2]:
from huggingface_hub import notebook_login
notebook_login()

In [3]:
import torch
import time
import json
import gc
import numpy as np

assert torch.cuda.is_available(), "No GPU"
props = torch.cuda.get_device_properties(0)
print(f"GPU: {props.name} ({props.total_memory / 1e9:.1f} GB)")
print(f"CUDA: {torch.version.cuda}  PyTorch: {torch.__version__}")

GPU: Tesla T4 (15.6 GB)
CUDA: 12.8  PyTorch: 2.10.0+cu128


In [4]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# 8B target in 4-bit (~5GB) + 1B draft in fp16 (~2GB) = ~7GB, fits T4
TARGET_MODEL = "meta-llama/Llama-3.1-8B"
DRAFT_MODEL  = "meta-llama/Llama-3.2-1B"   # same tokenizer family

MAX_NEW_TOKENS = 128
NUM_WARMUP     = 3
NUM_TRIALS     = 30
DRAFT_K_VALUES = [3, 5, 7, 10]
TEMPERATURES   = [0.0, 0.6, 0.8]

PROMPTS = [
    "The history of artificial intelligence began in the 1950s when researchers first",
    "In computer science, a hash table is a data structure that implements an associative",
    "The process of photosynthesis converts light energy into chemical energy through",
    'def fibonacci(n):\n    """Calculate the nth Fibonacci number."""\n    if n <= 1:\n        return n\n',
    "# Python implementation of binary search\ndef binary_search(arr, target):\n",
    "Summarize the following: Machine learning is a subset of artificial intelligence that focuses on building systems that learn from data. Machine learning algorithms use historical data as input to predict new output values. Machine learning is",
    "Write a detailed explanation of how TCP/IP networking works, starting from",
    "Explain the difference between a stack and a queue data structure. A stack",
    "The transformer architecture was introduced in the paper Attention is All You Need by Vaswani et al. in 2017. It replaced recurrent neural networks with self-attention mechanisms, enabling parallel processing of sequences. The key innovation was the multi-head attention mechanism, which allows the model to attend to different parts of the input simultaneously. Since then, transformers have become the foundation for",
    "Large language models are trained on massive datasets of text from the internet. During training, the model learns to predict the next token in a sequence given all previous tokens. This autoregressive training objective means that at inference time, the model generates text one token at a time, each conditioned on all previously generated tokens. This sequential nature creates a fundamental bottleneck because",
]

PROMPT_LABELS = [
    "general_1", "general_2", "general_3",
    "code_1", "code_2", "summarization",
    "instruction_1", "instruction_2",
    "long_ctx_1", "long_ctx_2",
]

print(f"Target: {TARGET_MODEL} (4-bit)")
print(f"Draft:  {DRAFT_MODEL} (fp16)")
print(f"Ratio:  ~1B / ~8B = 0.125")

Target: meta-llama/Llama-3.1-8B (4-bit)
Draft:  meta-llama/Llama-3.2-1B (fp16)
Ratio:  ~1B / ~8B = 0.125


In [5]:
# Load target model in 4-bit
print("Loading target model (4-bit)...")
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
)
target_model = AutoModelForCausalLM.from_pretrained(
    TARGET_MODEL, quantization_config=quant_config,
    torch_dtype=torch.float16, device_map="auto",
)
target_model.eval()

tokenizer = AutoTokenizer.from_pretrained(TARGET_MODEL)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

alloc_target = torch.cuda.memory_allocated() / 1e9
print(f"  Target loaded: {alloc_target:.2f} GB")

# Load draft model in fp16
print("Loading draft model (fp16)...")
draft_model = AutoModelForCausalLM.from_pretrained(
    DRAFT_MODEL, torch_dtype=torch.float16, device_map="auto",
)
draft_model.eval()

alloc_total = torch.cuda.memory_allocated() / 1e9
print(f"  Total: {alloc_total:.2f} GB (draft overhead: {alloc_total - alloc_target:.2f} GB)")

# Sanity check — baseline generation
print("\nSanity check (baseline)...")
test_inp = tokenizer("Hello", return_tensors="pt").to(target_model.device)
with torch.inference_mode():
    out = target_model.generate(**test_inp, max_new_tokens=10, pad_token_id=tokenizer.pad_token_id)
print(f"  Baseline: {tokenizer.decode(out[0], skip_special_tokens=True)}")

# Sanity check — assisted generation (no extra tokenizer args needed, same family)
print("Sanity check (assisted)...")
with torch.inference_mode():
    out = target_model.generate(
        **test_inp, max_new_tokens=10,
        assistant_model=draft_model,
        pad_token_id=tokenizer.pad_token_id,
    )
print(f"  Assisted: {tokenizer.decode(out[0], skip_special_tokens=True)}")
print("\nModels ready!")

Loading target model (4-bit)...


config.json:   0%|          | 0.00/826 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

...

  Total: 8.53 GB (draft overhead: 2.47 GB)

Sanity check (baseline)...


Passing `generation_config` together with generation-related arguments=({'min_new_tokens', 'max_new_tokens', 'use_cache'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=9) and `max_length`(=12) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `min_new_tokens` (=0) and `min_length`(=0) seem to have been set. `min_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  Baseline: Hello, I have a question. I have a 
Sanity check (assisted)...


Both `max_new_tokens` (=6) and `max_length`(=12) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `min_new_tokens` (=0) and `min_length`(=0) seem to have been set. `min_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  Assisted: Hello, I would like to see more pictures of the

Models ready!


In [6]:
# ============================================================
# Benchmark engine (identical to Phase 1)
# ============================================================
ALL_RESULTS = []
ALL_SUMMARIES = []

def timed_generate(model, inputs, gen_kwargs):
    torch.cuda.synchronize()
    t0 = time.perf_counter()
    with torch.inference_mode():
        output = model.generate(**inputs, **gen_kwargs)
    torch.cuda.synchronize()
    t1 = time.perf_counter()
    return output, t1 - t0

def run_benchmark(name, model, tokenizer, prompts, gen_kwargs,
                  num_warmup=NUM_WARMUP, num_trials=NUM_TRIALS):
    results = []
    for _ in range(num_warmup):
        inp = tokenizer(prompts[0], return_tensors="pt").to(model.device)
        with torch.inference_mode():
            model.generate(**inp, **gen_kwargs)
    torch.cuda.reset_peak_memory_stats()
    for trial in range(num_trials):
        pidx = trial % len(prompts)
        inp = tokenizer(prompts[pidx], return_tensors="pt").to(model.device)
        prompt_len = inp["input_ids"].shape[1]
        output, wall = timed_generate(model, inp, gen_kwargs)
        gen_tok = output.shape[1] - prompt_len
        results.append({
            "method": name, "prompt_idx": pidx, "prompt_tokens": prompt_len,
            "gen_tokens": gen_tok, "wall_sec": round(wall, 5),
            "tok_per_sec": round(gen_tok / wall, 2) if wall > 0 else 0,
            "peak_gpu_mb": round(torch.cuda.max_memory_allocated() / 1e6, 1),
        })
    return results

def summarize(results, baseline_tps=None):
    tps = [r["tok_per_sec"] for r in results]
    walls = [r["wall_sec"] for r in results]
    mean_tps = float(np.mean(tps))
    speedup = mean_tps / baseline_tps if baseline_tps else 1.0
    return {
        "method": results[0]["method"], "mean_tok_sec": round(mean_tps, 2),
        "median_tok_sec": round(float(np.median(tps)), 2),
        "std_tok_sec": round(float(np.std(tps)), 2),
        "p95_wall_sec": round(float(np.percentile(walls, 95)), 4),
        "speedup": round(speedup, 3), "n_trials": len(results),
        "peak_gpu_mb": max(r["peak_gpu_mb"] for r in results),
    }

def save_checkpoint(tag=""):
    suffix = f"_{tag}" if tag else ""
    with open(f"phase1b_results{suffix}.json", "w") as f:
        json.dump(ALL_RESULTS, f, indent=2)
    with open(f"phase1b_summaries{suffix}.json", "w") as f:
        json.dump(ALL_SUMMARIES, f, indent=2)
    print(f"  [saved: {len(ALL_RESULTS)} trials, {len(ALL_SUMMARIES)} summaries]")

def print_table(summaries):
    print(f"\n{'Method':<45} {'Mean T/s':>9} {'Med T/s':>9} {'Std':>7} {'Speedup':>8} {'GPU MB':>8}")
    print("-" * 95)
    for s in summaries:
        print(f"{s['method']:<45} {s['mean_tok_sec']:>9.2f} {s['median_tok_sec']:>9.2f} "
              f"{s['std_tok_sec']:>7.2f} {s['speedup']:>7.3f}x {s['peak_gpu_mb']:>8.0f}")

print("Engine ready.")

Engine ready.


## Baseline (8B 4-bit, greedy)

In [7]:
print("EXP: Baseline autoregressive (8B 4-bit, greedy)")
baseline_results = run_benchmark(
    "baseline_8B4bit_greedy", target_model, tokenizer, PROMPTS,
    dict(max_new_tokens=MAX_NEW_TOKENS, do_sample=False, pad_token_id=tokenizer.pad_token_id)
)
ALL_RESULTS.extend(baseline_results)
baseline_s = summarize(baseline_results)
ALL_SUMMARIES.append(baseline_s)
BASELINE_TPS = baseline_s["mean_tok_sec"]
print(f"Baseline: {BASELINE_TPS:.2f} tok/s")
save_checkpoint("baseline")

The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


EXP: Baseline autoregressive (8B 4-bit, greedy)
Baseline: 16.06 tok/s
  [saved: 30 trials, 1 summaries]


## Linear SD with 1B Draft (ratio=0.125) -- Sweep k

In [8]:
print("EXP: Linear SD 1B/8B (greedy, sweep k)")
for k in DRAFT_K_VALUES:
    print(f"\n--- k = {k} ---")
    results = run_benchmark(
        f"SD_1B8B_k{k}_greedy", target_model, tokenizer, PROMPTS,
        dict(
            max_new_tokens=MAX_NEW_TOKENS, assistant_model=draft_model,
            do_sample=False, pad_token_id=tokenizer.pad_token_id,
            num_assistant_tokens=k, num_assistant_tokens_schedule="constant",
        )
    )
    ALL_RESULTS.extend(results)
    s = summarize(results, BASELINE_TPS)
    ALL_SUMMARIES.append(s)
    print(f"  {s['mean_tok_sec']:.2f} tok/s  |  {s['speedup']:.3f}x")
save_checkpoint("after_k_sweep")

Both `max_new_tokens` (=20) and `max_length`(=144) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `min_new_tokens` (=0) and `min_length`(=0) seem to have been set. `min_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


EXP: Linear SD 1B/8B (greedy, sweep k)

--- k = 3 ---


Both `max_new_tokens` (=20) and `max_length`(=144) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `min_new_tokens` (=0) and `min_length`(=0) seem to have been set. `min_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=20) and `max_length`(=144) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `min_new_tokens` (=0) and `min_length`(=0) seem to have been set. `min_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_

  15.07 tok/s  |  0.938x

--- k = 5 ---


Both `max_new_tokens` (=20) and `max_length`(=144) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `min_new_tokens` (=0) and `min_length`(=0) seem to have been set. `min_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=20) and `max_length`(=144) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `min_new_tokens` (=0) and `min_length`(=0) seem to have been set. `min_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_

  15.10 tok/s  |  0.941x

--- k = 7 ---


Both `max_new_tokens` (=20) and `max_length`(=144) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `min_new_tokens` (=0) and `min_length`(=0) seem to have been set. `min_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=20) and `max_length`(=144) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `min_new_tokens` (=0) and `min_length`(=0) seem to have been set. `min_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_

  15.20 tok/s  |  0.946x

--- k = 10 ---


Both `max_new_tokens` (=20) and `max_length`(=144) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `min_new_tokens` (=0) and `min_length`(=0) seem to have been set. `min_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=20) and `max_length`(=144) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `min_new_tokens` (=0) and `min_length`(=0) seem to have been set. `min_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_

  14.89 tok/s  |  0.927x
  [saved: 150 trials, 5 summaries]


## Temperature Sensitivity (1B/8B, k=5)

In [9]:
print("EXP: Temperature sensitivity (1B/8B, k=5)")
TEMP_K = 5
for temp in TEMPERATURES:
    if temp == 0.0:
        continue
    print(f"\n--- Temperature = {temp} ---")

    base_res = run_benchmark(
        f"baseline_8B4bit_T{temp}", target_model, tokenizer, PROMPTS,
        dict(max_new_tokens=MAX_NEW_TOKENS, do_sample=True, temperature=temp,
             top_p=0.9, pad_token_id=tokenizer.pad_token_id)
    )
    ALL_RESULTS.extend(base_res)
    base_s = summarize(base_res)
    ALL_SUMMARIES.append(base_s)
    base_tps = base_s["mean_tok_sec"]

    sd_res = run_benchmark(
        f"SD_1B8B_k{TEMP_K}_T{temp}", target_model, tokenizer, PROMPTS,
        dict(
            max_new_tokens=MAX_NEW_TOKENS, assistant_model=draft_model,
            do_sample=True, temperature=temp, top_p=0.9,
            pad_token_id=tokenizer.pad_token_id,
            num_assistant_tokens=TEMP_K, num_assistant_tokens_schedule="constant",
        )
    )
    ALL_RESULTS.extend(sd_res)
    sd_s = summarize(sd_res, base_tps)
    ALL_SUMMARIES.append(sd_s)
    print(f"  Baseline T={temp}: {base_tps:.2f} tok/s")
    print(f"  SD 1B/8B k={TEMP_K} T={temp}: {sd_s['mean_tok_sec']:.2f} tok/s  |  {sd_s['speedup']:.3f}x")
save_checkpoint("after_temp")

EXP: Temperature sensitivity (1B/8B, k=5)

--- Temperature = 0.6 ---


Both `max_new_tokens` (=20) and `max_length`(=144) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `min_new_tokens` (=0) and `min_length`(=0) seem to have been set. `min_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=20) and `max_length`(=144) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `min_new_tokens` (=0) and `min_length`(=0) seem to have been set. `min_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_

  Baseline T=0.6: 16.30 tok/s
  SD 1B/8B k=5 T=0.6: 11.76 tok/s  |  0.721x

--- Temperature = 0.8 ---


Both `max_new_tokens` (=20) and `max_length`(=144) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `min_new_tokens` (=0) and `min_length`(=0) seem to have been set. `min_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=20) and `max_length`(=144) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `min_new_tokens` (=0) and `min_length`(=0) seem to have been set. `min_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_

  Baseline T=0.8: 16.12 tok/s
  SD 1B/8B k=5 T=0.8: 12.80 tok/s  |  0.794x
  [saved: 270 trials, 9 summaries]


## Per-Prompt Analysis

In [10]:
def get_prompt_avg(method_name):
    by_prompt = {}
    for r in ALL_RESULTS:
        if r["method"] == method_name:
            by_prompt.setdefault(r["prompt_idx"], []).append(r["tok_per_sec"])
    return {k: float(np.mean(v)) for k, v in by_prompt.items()}

baseline_pp = get_prompt_avg("baseline_8B4bit_greedy")
sd5_pp = get_prompt_avg("SD_1B8B_k5_greedy")

if sd5_pp:
    print(f"{'Prompt':<16} {'Baseline':>10} {'SD 1B/8B k=5':>14} {'Speedup':>10}")
    print("-" * 55)
    for pidx in sorted(baseline_pp.keys()):
        label = PROMPT_LABELS[pidx] if pidx < len(PROMPT_LABELS) else f"p{pidx}"
        b = baseline_pp.get(pidx, 1)
        sd = sd5_pp.get(pidx, 0)
        print(f"{label:<16} {b:>10.2f} {sd:>14.2f} {sd/b:>9.3f}x")
else:
    print("No SD k=5 results found")

Prompt             Baseline   SD 1B/8B k=5    Speedup
-------------------------------------------------------
general_1             13.80          10.88     0.788x
general_2             17.23           8.91     0.517x
general_3             16.11          13.44     0.834x
code_1                15.50          21.28     1.373x
code_2                15.99          23.20     1.451x
summarization         17.22          21.00     1.219x
instruction_1         15.89          15.99     1.006x
instruction_2         16.29          12.80     0.786x
long_ctx_1            16.72          13.26     0.793x
long_ctx_2            15.84          10.29     0.650x


## Summary

In [11]:
print_table(ALL_SUMMARIES)
save_checkpoint("final")

print("\n\nKEY COMPARISON:")
print("=" * 60)
print("Phase 1:  1B/3B  ratio=0.33 -> k=5 gave 0.977x SLOWDOWN")
print("Phase 1B: 1B/8B  ratio=0.125 -> k=5 gave ??? (check above)")
print()
print("Copy-paste for report:")
for s in ALL_SUMMARIES:
    print(f"  {s['method']}: {s['mean_tok_sec']} tok/s, speedup={s['speedup']}x, gpu={s['peak_gpu_mb']}MB")


Method                                         Mean T/s   Med T/s     Std  Speedup   GPU MB
-----------------------------------------------------------------------------------------------
baseline_8B4bit_greedy                            16.06     16.06    1.40   1.000x     8674
SD_1B8B_k3_greedy                                 15.07     13.43    4.68   0.938x     8722
SD_1B8B_k5_greedy                                 15.10     13.34    4.80   0.941x     8722
SD_1B8B_k7_greedy                                 15.20     13.14    5.12   0.946x     8722
SD_1B8B_k10_greedy                                14.89     12.91    4.92   0.927x     8722
baseline_8B4bit_T0.6                              16.30     16.11    0.72   1.000x     8674
SD_1B8B_k5_T0.6                                   11.76     10.10    4.97   0.721x     8723
baseline_8B4bit_T0.8                              16.12     16.08    1.05   1.000x     8674
SD_1B8B_k5_T0.8                                   12.80     10.93    4.67  